# 4 · Radar rain on dated landslides (dpc-radar + ispra-idrogeo)

rainfall-thresholds imports no other library: it takes a rain series and gives numbers. This
notebook joins it with its two sister libraries, the way Limen does:

- **ispra-idrogeo** gives the landslides with a date, from IdroGEO (ISPRA);
- **dpc-radar** gives the hourly rain at their place, from the radar of the Civil Protection;
- **rainfall-thresholds** cuts the rain into events and compares them with the Italian threshold.

It needs the network, and it is **not** run in CI. The DPC keeps radar frames for about five
months, so only recent landslides can be read.

> **A threshold exceeded is not an alert.** The official alert in Italy is the criticality
> bulletin of the Dipartimento della Protezione Civile. And this is **not a validation**: the
> Italian thresholds are fitted on rain gauges, the radar alone reads differently (in 2026 about
> 20 % more than the gauge-corrected «Merging», and not the same everywhere), and the sample is
> small and uneven.

Data: IdroGEO (ISPRA), CC BY-SA 4.0 · DPC radar, © Dipartimento della Protezione Civile,
CC BY-SA 4.0 · e-ITALICA v4, CC BY 4.0.

In [1]:
import asyncio
from collections import Counter
from datetime import UTC, date, datetime, time, timedelta

import numpy as np
from dpc_radar import ABSENT, RadarClient
from ispra_idrogeo import IdroGEO
from ispra_idrogeo.dated import dated_landslides

from rainfall_thresholds import ITALICA_T5, macroregion_of, reconstruct_events, strongest_event

SINCE = date(2026, 6, 1)  # the DPC keeps about five months of frames
HOURS = 48  # the rain read before the end of the landslide's day
LIMIT = 10  # the most recent landslides: every one costs up to 48 frame downloads

# ISTAT region codes -> e-ITALICA names, for macroregion_of
REGION = {
    1: "Piemonte",
    2: "Valle d'Aosta",
    3: "Lombardia",
    4: "Trentino-Alto Adige",
    5: "Veneto",
    6: "Friuli-Venezia Giulia",
    7: "Liguria",
    8: "Emilia-Romagna",
    9: "Toscana",
    10: "Umbria",
    11: "Marche",
    12: "Lazio",
    13: "Abruzzo",
    14: "Molise",
    15: "Campania",
    16: "Puglia",
    17: "Basilicata",
    18: "Calabria",
    19: "Sicilia",
    20: "Sardegna",
}
# Movements a rainfall threshold for shallow landslides is about: slides, flows, complex,
# areas of shallow landslides. Falls, sinkholes, lateral spreads and deep-seated
# deformations are left out.
RAINFALL_MOVEMENTS = {2, 4, 5, 7, 11}

## 1. Dated landslides, from IdroGEO

In [2]:
async def landslides():
    async with IdroGEO() as idrogeo:
        return await dated_landslides(
            idrogeo, sources=["eventi"], since=SINCE, max_interval_days=1, max_position_m=1000
        )


found = await landslides()
kept = [f for f in found if f.movement in RAINFALL_MOVEMENTS and f.lon is not None]
print(f"{len(found)} landslides since {SINCE}, dated to the day, point within 1 km")
print(f"{len(kept)} of them slides or flows (the rest are falls, sinkholes, n.d.)")
print(Counter(REGION[f.region] for f in kept).most_common())

42 landslides since 2026-06-01, dated to the day, point within 1 km
9 of them slides or flows (the rest are falls, sinkholes, n.d.)
[('Trentino-Alto Adige', 3), ('Lombardia', 3), ('Veneto', 2), ('Lazio', 1)]


## 2. The radar rain at each place

For each landslide, the hourly radar rain (SRT1) at its point over the 48 hours before the end
of its day. IdroGEO gives the day, not the hour: the window covers the whole day. A frame the
DPC no longer has, or a point the radar did not measure, is `None`: **not measured, not dry**.

In [3]:
recent = sorted(kept, key=lambda f: f.start)[-LIMIT:]


def hours_for(day):
    end = datetime.combine(day, time(23), tzinfo=UTC)
    return [end - timedelta(hours=h) for h in range(HOURS - 1, -1, -1)]


async def series_for(cases):
    wanted = sorted({t for f in cases for t in hours_for(f.start)})
    frames = {}
    limit = asyncio.Semaphore(8)
    async with RadarClient() as radar:

        async def one(t):
            async with limit:
                frames[t] = await radar.at("SRT1", t)

        await asyncio.gather(*(one(t) for t in wanted))
    out = {}
    for f in cases:
        rows = []
        for t in hours_for(f.start):
            frame = frames[t]
            value = None if frame is ABSENT else frame.sample([(f.lon, f.lat)])[0]
            rows.append((t, None if value is None or not np.isfinite(value) else float(value)))
        out[f.id] = rows
    gone = sum(v is ABSENT for v in frames.values())
    print(f"{len(wanted)} frames asked, {gone} no longer at the DPC")
    return out


series = await series_for(recent)

264 frames asked, 0 no longer at the DPC


## 3. Against the Italian threshold

In [4]:
rows = []
for f in recent:
    name = REGION[f.region]
    t = ITALICA_T5[macroregion_of(name) or "italy"]
    events = reconstruct_events(
        series[f.id], no_rain_break_hours=6, min_event_mm=2.0, source="DPC SRT1"
    )
    e = strongest_event(events, t)
    missing = sum(mm is None for _, mm in series[f.id])
    if e is None:
        rows.append((f.id, f.start, name, None, None, None, missing))
    else:
        x = t.exceedance(e.intensity_mm_h, e.duration_hours)
        rows.append((f.id, f.start, name, e.total_mm, e.duration_hours, x, missing))

head = ("landslide", "day", "region", "rain", "hours", "exceed.", "not meas.")
print(f"{head[0]:<14}{head[1]:<12}{head[2]:<22}{head[3]:>9}{head[4]:>7}{head[5]:>9}{head[6]:>11}")
for id_, day, name, mm, d, x, missing in rows:
    if mm is None:
        what = "no rain event" if missing < HOURS else "not measured"
        print(f"{id_:<14}{day!s:<12}{name:<22}{what:>25}{missing:>11}")
    else:
        print(f"{id_:<14}{day!s:<12}{name:<22}{mm:>7.1f}mm{d:>7.0f}{x:>+9.2f}{missing:>11}")

landslide     day         region                     rain  hours  exceed.  not meas.
EIT202600065  2026-06-27  Trentino-Alto Adige      26.1mm      4    +0.32          0
EIT202600066  2026-06-28  Trentino-Alto Adige      44.1mm      7    +0.43          0
EIT202600076  2026-08-02  Trentino-Alto Adige       9.5mm      5    -0.17          0
EIT202600109  2026-08-02  Veneto                   45.5mm      7    +0.44          0
EIT202600078  2026-08-11  Lazio                             no rain event          0
EIT202600083  2026-08-20  Lombardia               102.6mm     26    +0.52          0
EIT202600086  2026-08-20  Lombardia               153.4mm     28    +0.68          0
EIT202600089  2026-08-20  Lombardia                79.4mm     25    +0.41          0
EIT202600097  2026-09-10  Veneto                   78.0mm     45    +0.28          0


In [5]:
measured = [r for r in rows if r[3] is not None]
above = sum(r[5] > 0 for r in measured)
print(f"{above} of {len(measured)} landslides with a rain event had it above T5 on radar rain")
print(f"{len(rows) - len(measured)} with no rain event or no measured rain in the {HOURS} hours")

7 of 8 landslides with a rain event had it above T5 on radar rain
1 with no rain event or no measured rain in the 48 hours


## What this says, and what it does not

- It shows the **join**: three libraries, each on its own, put together in a few lines. Limen
  does the same in its scoring and backtests.
- It is **not a measure** of the threshold on radar rain. Ten recent landslides, mostly in the
  north, are an anecdote: IdroGEO dates to the day less than 1 % of the landslides in 16
  regions. The measure, on all the landslides since 2018 and on the radar history, runs in
  Limen ([#8](https://github.com/agent-engineering-studio/rainfall-thresholds/issues/8)).
- A landslide under the threshold here may be a landslide the radar read low, one triggered by
  the weeks before rather than the day, or one the threshold misses. The notebook cannot tell
  which.
- Nothing here is an alert.